**Question 1:** How do you select rows from a DataFrame where **any value in the row** exceeds a given threshold?

**Question 2:** How do you sort a DataFrame by column `A` ascending and column `B` descending simultaneously?

**Question 3:** How do you concatenate two DataFrames vertically and reset the index?

**Question 4:** What is the difference between removing outliers and capping them? When would you choose each approach?

**Question 5:** You call `df.fillna(df.mean())` on a DataFrame that has both numeric and categorical columns. What happens, and is there a better approach?


**Q1:** `df[df.max(axis=1) > threshold]` — computes the row-wise maximum and filters rows where it exceeds the threshold.

**Q2:** `df.sort_values(['A', 'B'], ascending=[True, False])`

**Q3:** `pd.concat([df1, df2], ignore_index=True)` — `ignore_index=True` resets the index from 0.

**Q4:** **Removing** permanently deletes the outlier rows, which can reduce your dataset size significantly. **Capping** (clipping to a max/min threshold) preserves the row but limits the extreme value. Remove when outliers are clear data errors; cap when they're legitimate but distort analysis.

**Q5:** `.fillna(df.mean())` will only work on numeric columns — categorical columns will be ignored and remain NaN. A better approach: `df.fillna({'numeric_col': df['numeric_col'].mean(), 'cat_col': df['cat_col'].mode()[0]})`

## Part 2: Practical Challenge (30–45 min)

### Scenario: "The Patient Records Audit"

A hospital has given you a messy export of patient records. Your job is to clean it and produce a summary report before it can be used for any analysis.

---


In [6]:
### Challenge 1: The Health Check

import pandas as pd
import numpy as np

# Simulated messy hospital data
data = {
    'patient_id': [101, 102, 103, 102, 104, 105, 106],
    'name': ['Alice', 'Bob', 'Charlie', 'Bob', 'Diana', None, 'Frank'],
    'age': [34, 28, 150, 28, 45, 62, 29],  # Note: 150 is impossible
    'blood_type': ['A+', 'O-', 'B+', 'O-', None, 'AB+', 'A-'],
    'ward': ['Cardiology', 'neurology', 'Cardiology', 'neurology', 'Oncology', 'Cardiology', 'neurology'],
    'admission_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-01-20', '2024-02-10', '2024-02-15', '2024-03-01'],
    'days_admitted': [5, 3, 8, 3, None, 12, 4]
}

df = pd.DataFrame(data)


**Tasks:**
1. Run `.info()` — which columns have missing values?
2. Run `.describe()` — identify the outlier in the `age` column.
3. Use `.isnull().sum()` to count missing values per column.
4. Use `.duplicated().sum()` to count duplicate rows.

<details>
<summary>💡 Hint</summary>

A duplicated row is one where **all** column values are identical. You can also check specific columns: `df.duplicated(subset=['patient_id']).sum()`

</details>


### Challenge 2: Clean the Data

**Tasks:**
1. Remove the duplicate row (keep the first occurrence).
2. Fix the impossible age (150) — replace it with the column median.
3. Standardise the `ward` column — all values should be title case (`'Cardiology'`, not `'cardiology'`).
4. Fill the missing `blood_type` with the string `'Unknown'`.
5. Fill missing `days_admitted` with the column median.

<details>
<summary>✅ Solution</summary>

In [2]:
# 1. Remove duplicates
df = df.drop_duplicates()

# 2. Fix impossible age
median_age = df.loc[df['age'] <= 120, 'age'].median()
df.loc[df['age'] > 120, 'age'] = median_age

# 3. Standardise ward names
df['ward'] = df['ward'].str.title()

# 4. Fill missing blood type
df['blood_type'] = df['blood_type'].fillna('Unknown')

# 5. Fill missing days admitted
df['days_admitted'] = df['days_admitted'].fillna(df['days_admitted'].median())

print(df)


   patient_id     name  age blood_type        ward admission_date  \
0         101    Alice   34         A+  Cardiology     2024-01-15   
1         102      Bob   28         O-   Neurology     2024-01-20   
2         103  Charlie   34         B+  Cardiology     2024-02-01   
4         104    Diana   45    Unknown    Oncology     2024-02-10   
5         105     None   62        AB+  Cardiology     2024-02-15   
6         106    Frank   29         A-   Neurology     2024-03-01   

   days_admitted  
0            5.0  
1            3.0  
2            8.0  
4            5.0  
5           12.0  
6            4.0  


### Challenge 3: Transform & Summarise

**Tasks:**
1. Convert `admission_date` to datetime type.
2. Add a `risk_level` column using `.apply()`: age > 60 → `'High'`; age 40–60 → `'Medium'`; under 40 → `'Low'`.
3. Use `.value_counts()` to count patients per `ward`.
4. Compute the average `days_admitted` per `ward`.


In [3]:
# 1. Convert date
df['admission_date'] = pd.to_datetime(df['admission_date'])

# 2. Risk level
def risk_level(age):
    if age > 60:
        return 'High'
    elif age >= 40:
        return 'Medium'
    else:
        return 'Low'

df['risk_level'] = df['age'].apply(risk_level)

# 3. Patients per ward
print(df['ward'].value_counts())

# 4. Average days per ward
print(df.groupby('ward')['days_admitted'].mean().round(1))

Cardiology    3
Neurology     2
Oncology      1
Name: ward, dtype: int64
ward
Cardiology    8.3
Neurology     3.5
Oncology      5.0
Name: days_admitted, dtype: float64


### 🏆 Stretch Challenge (Optional)

Export the cleaned DataFrame to a CSV file, then reload it and verify:
- All data types are preserved (especially the datetime column).
- Row count matches what you expect.

> **Hint:** When reloading, you'll need to parse the date column again with `parse_dates=['admission_date']`.

<details>
<summary>✅ Stretch Challenge Solution</summary>

In [4]:
df.to_csv('cleaned_patients.csv', index=False)

# Reload
df_reloaded = pd.read_csv('cleaned_patients.csv', parse_dates=['admission_date'])

# Verify
print(df_reloaded.info())
print(f"Row count: {len(df_reloaded)}")  # Should match len(df) after cleaning
print(df_reloaded.dtypes)                # admission_date should be datetime64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   patient_id      6 non-null      int64         
 1   name            5 non-null      object        
 2   age             6 non-null      int64         
 3   blood_type      6 non-null      object        
 4   ward            6 non-null      object        
 5   admission_date  6 non-null      datetime64[ns]
 6   days_admitted   6 non-null      float64       
 7   risk_level      6 non-null      object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 512.0+ bytes
None
Row count: 6
patient_id                 int64
name                      object
age                        int64
blood_type                object
ward                      object
admission_date    datetime64[ns]
days_admitted            float64
risk_level                object
dtype: objec


**What to watch for:** Without `parse_dates=['admission_date']`, the date column reloads as `object` (string). CSV format doesn't store data type information — you always need to re-specify date columns on reload. This is a common source of bugs in data pipelines.
